# kiji-inspector quickstart — Qwen3.6-35B-A3B SAEs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dataiku/kiji-inspector/blob/main/demo/quickstart_g4.ipynb)

Interpret a residual-stream activation of `Qwen/Qwen3.6-35B-A3B` with the published
[layer-sweep SAEs](https://huggingface.co/575-lab/kiji-inspector-Qwen-Qwen3.6-35B-A3B) (layers 12/20/28/36).

> **Hardware:** the model loads in bf16 (~70 GB of weights) — you need a ~96 GB GPU such as an
> **RTX Pro 6000**, A100-80GB, or H100. Standard Colab T4/A100-40GB instances are too small.

In [ ]:
!pip install -U -q kiji-inspector transformers accelerate

## Build the agent prompt and extract the decision-token activation

The SAEs were trained on activations at the **decision token**: an agent system prompt with a
tool list, a user request, and the assistant prefill `"I'll use the "` — the last token is where
the model commits to a tool choice. We reproduce that format exactly (including
`enable_thinking=False`, so the prefill sits at the final-answer position, not inside a
`<think>` block).

We run a real contrastive pair from the training distribution (`tool_selection` scenario,
`internal_vs_external` contrast type) — two phrasings of the same question, one that should
route to internal company docs and one that should route to the public web. This pair is one
of the rare cases where the two prompts activate **near-disjoint** top feature sets, rather
than the more common pattern of shared features with shifted magnitudes.

> **Layer indexing:** the SAEs are keyed by vLLM auxiliary hidden-state IDs
> (`eagle_aux_hidden_state_layer_ids`), and vLLM aux ID `k` equals the output of HF transformers
> decoder layer `k − 1` (verified in
> [residual-stream-model-comparisons](https://github.com/Davidnet/residual-stream-model-comparisons),
> cosine ≈ 0.99997). So for the SAE at `layer_28` we hook `model.model.language_model.layers[27]`.

In [1]:
import torch
from transformers import AutoModelForMultimodalLM, AutoProcessor

MODEL_ID = "Qwen/Qwen3.6-35B-A3B"
SAE_LAYER = 28  # vLLM aux hidden-state ID (SAE repo layer_28)
HF_LAYER_INDEX = SAE_LAYER - 1  # vLLM aux ID k == HF decoder layer k-1
PREFILL = "I'll use the "

SYSTEM_PROMPT = "You are a helpful assistant. Choose the best tool for each request."

TOOLS = [
    ("internal_search", "Search internal company documentation"),
    ("web_search", "Search the public web"),
    ("file_read", "Read a local file"),
    ("file_write", "Write or update a local file"),
    ("database_query", "Query a SQL database"),
    ("api_call", "Call an external REST API"),
    ("code_execute", "Execute code in a sandbox"),
    ("delegate_agent", "Delegate to a sub-agent for complex tasks"),
]

# A contrastive pair from the SAE training distribution: the same question about incident
# response plans, phrased to route to internal docs vs. the public web.
PROMPTS = {
    "anchor (expects internal_search)": "What is our company\u2019s incident response plan as outlined in internal security docs?",
    "contrast (expects web_search)": "What is the standard incident response plan for companies according to public cybersecurity resources?",
}

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID, dtype="auto", device_map="auto", low_cpu_mem_usage=True
)
model.eval()
layers = model.model.language_model.layers
input_device = model.model.language_model.embed_tokens.weight.device
print(f"{len(layers)} decoder layers | hooking layers[{HF_LAYER_INDEX}] for SAE layer_{SAE_LAYER}")

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/1026 [00:00<?, ?it/s]

40 decoder layers | hooking layers[27] for SAE layer_28


In [2]:
def build_decision_prompt(user_request: str) -> str:
    """Mirror kiji_inspector.extraction.extractor.build_agent_prompt."""
    tool_descriptions = "\n".join(f"- {name}: {desc}" for name, desc in TOOLS)
    messages = [
        {
            "role": "system",
            "content": (
                f"{SYSTEM_PROMPT}\n\n"
                f"Available tools:\n{tool_descriptions}\n\n"
                f"When you decide to use a tool, respond with the tool name."
            ),
        },
        {"role": "user", "content": user_request},
    ]
    formatted = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    if formatted.rstrip().endswith("<think>"):  # template ignored enable_thinking
        formatted = formatted.rstrip() + "\n\n</think>\n\n"
    return formatted + PREFILL


def decision_token_activation(user_request: str) -> torch.Tensor:
    """Residual-stream output of layers[HF_LAYER_INDEX] at the last (decision) token."""
    text = build_decision_prompt(user_request)
    inputs = processor.tokenizer(text, return_tensors="pt", add_special_tokens=False)
    inputs = {k: v.to(input_device) for k, v in inputs.items()}

    captured = {}

    def hook(_module, _inputs, output):
        hidden = output[0] if isinstance(output, tuple) else output
        captured["h"] = hidden.detach()

    handle = layers[HF_LAYER_INDEX].register_forward_hook(hook)
    try:
        with torch.inference_mode():
            model(**inputs, use_cache=False)
    finally:
        handle.remove()
    return captured["h"][0, -1].float().cpu()


activations = {label: decision_token_activation(req) for label, req in PROMPTS.items()}
{label: tuple(act.shape) for label, act in activations.items()}

{'anchor (expects internal_search)': (2048,),
 'contrast (expects web_search)': (2048,)}

## Load the SAE and describe the active features

`SAE.from_pretrained` resolves the base model through the built-in registry. Activations are
rescaled by `sae.rms_scale` — a single global dataset-level RMS constant used at training time
(not per-vector normalization) — before encoding.

In [3]:
from kiji_inspector import SAE

sae, feature_descriptions = SAE.from_pretrained(base_model=MODEL_ID, layer=SAE_LAYER)
print(
    f"d_model={sae.d_model} d_sae={sae.d_sae} rms_scale={sae.rms_scale:.4f} "
    f"labeled_features={len(feature_descriptions)}"
)

results = {
    label: sae.describe(act / sae.rms_scale, feature_descriptions)
    for label, act in activations.items()
}

d_model=2048 d_sae=8192 rms_scale=0.0891 labeled_features=62


In [4]:
for label, request in PROMPTS.items():
    print(f"=== {label} ===")
    print(f'"{request}"')
    for feature_id, desc, activation in results[label]:
        print(f"  Feature {feature_id} | activation {activation:.2f}")
        if isinstance(desc, dict):
            print(f"    {desc.get('label', 'N/A')}")
            print(f"    {desc.get('description', 'N/A')}")
        else:
            print(f"    {desc}")
    print()

=== anchor (expects internal_search) ===
"What is our company’s incident response plan as outlined in internal security docs?"
  Feature 5925 | activation 21.18
    Inquiry about system specifications and constraints
    This feature activates when the user asks for factual details regarding API codes, supported file formats, operational limits, or system capabilities.
  Feature 7534 | activation 20.67
    Troubleshooting and Account Recovery
    Detects queries involving debugging technical failures, diagnosing root causes of system errors, or recovering access to locked accounts.
  Feature 1776 | activation 13.55
    Retrieval of current official documentation
    This feature activates when the user requests the latest version, status, or specific content of internal documents, schedules, or technical references.
  Feature 891 | activation 11.43
    Quality Control and Defect Analysis
    This feature detects queries related to monitoring manufacturing quality, analyzing defect rate